In [1]:
import sys
print(sys.executable)

c:\Users\raich\anaconda3\envs\major_env\python.exe


In [2]:
import sys
!{sys.executable} -m pip install faiss-cpu

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import faiss

BASE_DIR = Path(
    r"C:\Users\raich\Desktop\major\Healthcare_Dataset_Preparation"
)

EMBEDDINGS_PATH = (
    BASE_DIR
    / "outputs"
    / "embeddings"
    / "knowledge_base_embeddings.npy"
)

METADATA_PATH = (
    BASE_DIR
    / "outputs"
    / "embeddings"
    / "embedding_metadata.csv"
)

RETRIEVAL_PATH = (
    BASE_DIR
    / "data"
    / "processed"
    / "knowledge"
    / "knowledge_base_retrieval.csv"
)

FAISS_DIR = (
    BASE_DIR
    / "outputs"
    / "faiss"
)

FAISS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

embeddings = np.load(
    EMBEDDINGS_PATH
)

metadata = pd.read_csv(
    METADATA_PATH
)

retrieval_df = pd.read_csv(
    RETRIEVAL_PATH
)

print("Embeddings:", embeddings.shape)
print("Metadata:", metadata.shape)
print("Retrieval data:", retrieval_df.shape)

Embeddings: (15979, 768)
Metadata: (15979, 3)
Retrieval data: (15979, 6)


In [4]:
assert len(embeddings) == len(metadata)
assert len(metadata) == len(retrieval_df)

assert metadata["document_id"].is_unique
assert retrieval_df["document_id"].is_unique

assert (
    metadata["document_id"].values
    == retrieval_df["document_id"].values
).all()

print("✓ Embeddings and metadata are aligned.")
print("Documents:", len(metadata))
print("Dimensions:", embeddings.shape[1])

✓ Embeddings and metadata are aligned.
Documents: 15979
Dimensions: 768


In [5]:
normalize_embeddings=True
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(
    dimension
)

index.add(
    embeddings.astype("float32")
)

print("FAISS index created.")
print("Number of vectors:", index.ntotal)
print("Dimension:", index.d)

FAISS index created.
Number of vectors: 15979
Dimension: 768


In [6]:
FAISS_INDEX_PATH = (
    FAISS_DIR
    / "knowledge_base.index"
)

faiss.write_index(
    index,
    str(FAISS_INDEX_PATH)
)

print(
    "Saved:",
    FAISS_INDEX_PATH
)

Saved: C:\Users\raich\Desktop\major\Healthcare_Dataset_Preparation\outputs\faiss\knowledge_base.index


In [7]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = (
    "pritamdeka/"
    "BioBERT-mnli-snli-scinli-scitail-mednli-stsb"
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL
)

print("Embedding model loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded.


In [9]:
def retrieve_documents(
    query,
    top_k=5
):
    
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(
        query_embedding.astype("float32"),
        top_k
    )

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        result = {
            "document_id": int(
                metadata.iloc[idx]["document_id"]
            ),
            "score": float(score),
            "source_dataset": metadata.iloc[idx][
                "source_dataset"
            ],
            "prompt": retrieval_df.iloc[idx][
                "prompt"
            ],
            "response": retrieval_df.iloc[idx][
                "response"
            ]
        }

        results.append(result)

    return pd.DataFrame(results)

In [10]:
queries = [
    "What are the symptoms of monkeypox?",
    "What is Marfan syndrome?",
    "What are the symptoms of Kallmann syndrome?",
    "What is vitamin K deficiency?",
    "What causes diabetes?"
]

for query in queries:

    print("\n" + "=" * 80)
    print("QUERY:", query)
    print("=" * 80)

    results = retrieve_documents(
        query,
        top_k=5
    )

    display(results)


QUERY: What are the symptoms of monkeypox?


,document_id,score,source_dataset,prompt,response
0,0,0.702168,MedQuAD,What is (are) Monkeypox Virus Infections ?,Monkeypox is a rare viral disease. It occurs m...
1,11910,0.473750,MedQuAD,What are the symptoms of Moyamoya disease ?,What are the signs and symptoms of Moyamoya di...
2,12720,0.466587,MedQuAD,What is (are) Mumps ?,Mumps is an illness caused by the mumps virus....
3,5817,0.462816,MedQuAD,What is (are) Malaria ?,Malaria is a serious and sometimes fatal disea...
4,8164,0.461914,MedQuAD,What is (are) Smallpox ?,Smallpox is a disease caused by the Variola ma...



QUERY: What is Marfan syndrome?


,document_id,score,source_dataset,prompt,response
0,4447,0.653252,MedQuAD,What are the treatments for Marfan syndrome ?,These resources address the diagnosis or manag...
1,6144,0.638887,MedQuAD,What are the symptoms of Marfan Syndrome ?,Marfan syndrome can affect many parts of the b...
2,8785,0.592206,MedQuAD,How to diagnose Marfan Syndrome ?,Your doctor will diagnose Marfan syndrome base...
3,11505,0.580755,MedQuAD,How many people are affected by Marfan syndrome ?,The incidence of Marfan syndrome is approximat...
4,5268,0.578339,MedQuAD,What is (are) Marfan Syndrome ?,Marfan syndrome is a disorder that affects con...



QUERY: What are the symptoms of Kallmann syndrome?


,document_id,score,source_dataset,prompt,response
0,15226,0.711125,MedQuAD,What are the symptoms of Kallmann syndrome ?,What are the signs and symptoms of Kallmann sy...
1,14615,0.657105,MedQuAD,Is Kallmann syndrome inherited ?,How is Kallmann syndrome inherited? Kallmann s...
2,6406,0.649028,MedQuAD,What are the symptoms of Kallmann syndrome 3 ?,What are the signs and symptoms of Kallmann sy...
3,3523,0.646201,MedQuAD,What is (are) Kallmann syndrome ?,Kallmann syndrome (KS) is a condition characte...
4,15568,0.630627,MedQuAD,How many people are affected by Kallmann syndr...,Kallmann syndrome is estimated to affect 1 in ...



QUERY: What is vitamin K deficiency?


,document_id,score,source_dataset,prompt,response
0,1,0.588927,MedQuAD,Do you have information about Vitamin K,Summary : Vitamins are substances that your bo...
1,999,0.530130,PubMedQA,Treatment of vitamin D deficiency in CKD patie...,Current K/DOQI guidelines are inadequate for c...
2,6115,0.526665,MedQuAD,Do you have information about Vitamin D,Summary : Vitamins are substances that your bo...
3,6658,0.518429,PubMedQA,Is vitamin D deficiency a feature of pediatric...,Our data showed no difference in 25(OH) D leve...
4,15094,0.518215,PubMedQA,Is vitamin D insufficiency or deficiency relat...,These first data show that a vitamin D3 defici...



QUERY: What causes diabetes?


,document_id,score,source_dataset,prompt,response
0,14630,0.686565,MedQuAD,What to do for Causes of Diabetes ?,- Diabetes is a complex group of diseases with...
1,5588,0.627018,MedQuAD,Is type 1 diabetes inherited ?,A predisposition to develop type 1 diabetes is...
2,5471,0.619960,MedQuAD,What to do for Monogenic Forms of Diabetes: Ne...,- Mutations in single genes can cause rare for...
3,14592,0.616073,MedQuAD,What causes Diabetic Neuropathies: The Nerve D...,The causes are probably different for differen...
4,3600,0.608402,MedQuAD,What causes Diabetes ?,Type 1 diabetes is an autoimmune disease. In a...


In [11]:
query = "What are the symptoms of monkeypox?"

results = retrieve_documents(
    query,
    top_k=5
)

for i, row in results.iterrows():

    print("\n" + "-" * 70)

    print(
        "Rank:",
        i + 1
    )

    print(
        "Score:",
        round(row["score"], 4)
    )

    print(
        "Source:",
        row["source_dataset"]
    )

    print(
        "Prompt:",
        row["prompt"]
    )

    print(
        "Response:",
        row["response"][:700]
    )


----------------------------------------------------------------------
Rank: 1
Score: 0.7022
Source: MedQuAD
Prompt: What is (are) Monkeypox Virus Infections ?
Response: Monkeypox is a rare viral disease. It occurs mostly in central and western Africa. Wild rodents and squirrels carry it, but it is called monkeypox because scientists saw it first in lab monkeys. In 2003, it was reported in prairie dogs and humans in the U.S.     Centers for Disease Control and Prevention

----------------------------------------------------------------------
Rank: 2
Score: 0.4737
Source: MedQuAD
Prompt: What are the symptoms of Moyamoya disease ?
Response: What are the signs and symptoms of Moyamoya disease? The Human Phenotype Ontology provides the following list of signs and symptoms for Moyamoya disease. If the information is available, the table below includes how often the symptom is seen in people with this condition. You can use the MedlinePlus Medical Dictionary to look up the definitions for 

In [12]:
results = retrieve_documents(
    "What are the symptoms of monkeypox?",
    top_k=10
)

print(
    results[
        [
            "document_id",
            "score",
            "source_dataset",
            "prompt"
        ]
    ].to_string(index=False)
)

 document_id    score source_dataset                                                                                              prompt
           0 0.702168        MedQuAD                                                          What is (are) Monkeypox Virus Infections ?
       11910 0.473750        MedQuAD                                                         What are the symptoms of Moyamoya disease ?
       12720 0.466587        MedQuAD                                                                               What is (are) Mumps ?
        5817 0.462816        MedQuAD                                                                             What is (are) Malaria ?
        8164 0.461914        MedQuAD                                                                            What is (are) Smallpox ?
        4800 0.460872        MedQuAD                                                                  What are the symptoms of Malaria ?
       10565 0.460460        MedQuAD     

In [13]:
security_queries = [
    "What is prompt injection in a healthcare LLM?",
    "How can patient information be protected from an AI system?",
    "What is PHI?",
    "How can jailbreak attacks affect medical AI?"
]

for query in security_queries:

    print("\n" + "=" * 80)
    print("QUERY:", query)
    print("=" * 80)

    results = retrieve_documents(
        query,
        top_k=5
    )

    display(
        results[
            [
                "score",
                "source_dataset",
                "prompt"
            ]
        ]
    )


QUERY: What is prompt injection in a healthcare LLM?


,score,source_dataset,prompt
0,0.491291,MedQuAD,How to diagnose Stroke ?
1,0.450909,MedQuAD,What is the outlook for Oxygen Therapy ?
2,0.449574,MedQuAD,How to diagnose Urinary Incontinence ?
3,0.442424,MedQuAD,What are the treatments for Transient Ischemic...
4,0.437729,MedQuAD,How to diagnose What I need to know about Hepa...



QUERY: How can patient information be protected from an AI system?


,score,source_dataset,prompt
0,0.569784,MedQuAD,Do you have information about Patient Rights
1,0.502464,MedQuAD,Do you have information about Personal Health ...
2,0.462620,PubMedQA,Informed consent for total hip arthroplasty: d...
3,0.449535,MedQuAD,Do you have information about Medical Device S...
4,0.445678,MedQuAD,Do you have information about Patient Safety



QUERY: What is PHI?


,score,source_dataset,prompt
0,0.401253,MedQuAD,What are the treatments for Bell's palsy ?
1,0.397172,MedQuAD,What causes Bell's palsy ?
2,0.388715,MedQuAD,How to diagnose Phacomatosis pigmentovascularis ?
3,0.373193,MedQuAD,How many people are affected by blepharophimos...
4,0.372359,MedQuAD,What are the treatments for Pulmonary Hyperten...



QUERY: How can jailbreak attacks affect medical AI?


,score,source_dataset,prompt
0,0.452469,MedQuAD,How to diagnose ARDS ?
1,0.444295,MedQuAD,How to diagnose Alkhurma Hemorrhagic Fever (AH...
2,0.439281,PubMedQA,Discharging patients earlier from Winnipeg hos...
3,0.426421,PubMedQA,Israeli hospital preparedness for terrorism-re...
4,0.426321,PubMedQA,Does a special interest in laparoscopy affect ...


In [14]:
retrieval_test_path = (
    FAISS_DIR
    / "retrieval_test_results.csv"
)

all_results = []

for query in queries:

    results = retrieve_documents(
        query,
        top_k=5
    )

    results.insert(
        0,
        "query",
        query
    )

    all_results.append(results)

retrieval_test_df = pd.concat(
    all_results,
    ignore_index=True
)

retrieval_test_df.to_csv(
    retrieval_test_path,
    index=False
)

print(
    "Saved:",
    retrieval_test_path
)

Saved: C:\Users\raich\Desktop\major\Healthcare_Dataset_Preparation\outputs\faiss\retrieval_test_results.csv


In [ ]:
display(
    evaluation_results_with_source.groupby(
        "source_dataset"
    )[["top_1", "top_3", "top_5"]].mean()
)

: 